# Phase 6b · Sensitivity analyses and data checks for the manuscript

CPU only, from the existing Phase 3 outputs (`output/phase3/xfi_<model>/results_main.csv`, `results_base.csv`) and the DICOM headers. No model is retrained and no map is recomputed.

| Section | Question | Output |
|---|---|---|
| 1 | Which DICOM attributes does the preprocessing meet (photometric interpretation, LUTs, padding, bit depth)? | `R14_dicom_headers.csv` |
| 2 | Does the cross-patient reference change when both patients have the same predicted class? | `R15_crosspatient_by_class.csv` |
| 3 | Does the Grad-CAM vs Eigen-CAM comparison change when only patients whose prediction did not flip are used? | `R16_grad_vs_eigen_noflip.csv` |
| 4 | Patient-cluster bootstrap CIs for the descriptive validity correlations | `R17_validity_ci.csv` |
| 5 | Does the XFI, and the Grad-CAM vs Eigen-CAM comparison, depend on the prediction outcome on the clean image (TP, FN, TN, FP)? | `R18_xfi_by_outcome.csv` |

All CIs: patient bootstrap, percentile 95%. Tables go to `output/phase4/`.

## Configuration cell

`B_CORR` is the number of cluster-bootstrap resamples for the correlations in section 4 (one Spearman correlation over all patient-condition rows per resample). 2000 takes a few minutes per model; lower it for a quick test.

In [ ]:
import sys, json, hashlib, time
from pathlib import Path
import numpy as np, pandas as pd, yaml
from scipy.stats import spearmanr, rankdata

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from r1lib import stats as ST
CFG = yaml.safe_load(open(ROOT / "configs" / "config.yaml"))
SEED, B = CFG["project"]["seed"], CFG["metrics"]["bootstrap_B"]
B_CORR = 2000
OUT = ROOT / "output" / "phase4"; OUT.mkdir(parents=True, exist_ok=True)
IMG_DIR = ROOT / CFG["paths"]["image_dir"]
MODEL_ORDER = ["A", "Arep", "Anorm", "B"]
MODELS = [m for m in MODEL_ORDER if (ROOT / "output" / "phase3" / f"xfi_{m}" / "results_main.csv").exists()]
METHODS = ["GradCAM", "EigenCAM"]
TYPES, LEVELS = ["Gaussian", "Blur", "SaltPepper"], [1, 2, 3]
PRODUCED = []

def sha256(p):
    h = hashlib.sha256(); h.update(Path(p).read_bytes()); return h.hexdigest()
def save_table(df, name):
    p = OUT / f"{name}.csv"; df.to_csv(p, index=False); PRODUCED.append(p); return df
def ci(x, stat=np.mean):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    return ST.boot_ci_stat(x, stat, B=B, seed=SEED) if len(x) else (np.nan, np.nan, np.nan)
print("models:", MODELS, "| B =", B, "| B_CORR =", B_CORR, "| DICOM dir exists:", IMG_DIR.exists())

## Load

One row per patient x method x condition in `main`; baselines in `base`. The hashes are checked against the Phase 3 manifests.

In [ ]:
main, base = {}, {}
for M in MODELS:
    d = ROOT / "output" / "phase3" / f"xfi_{M}"
    man = json.loads((d / "manifest.json").read_text())
    assert sha256(d / "results_main.csv") == man["results_main_sha256"], f"{M}: results_main hash mismatch"
    m = pd.read_csv(d / "results_main.csv"); b = pd.read_csv(d / "results_base.csv")
    m["abs_d_iou"] = (m.iou_rel90_c - m.iou_rel90_n).abs()
    main[M], base[M] = m, b
    print(f"{M}: {len(m)} main rows, {len(b)} baseline rows")

## 1 · DICOM headers

The preprocessing reads `pixel_array` and min-max scales each image to [0, 255]; no modality LUT, VOI LUT or inversion is applied. This is correct only if every image is MONOCHROME2, has no rescale or VOI transform that would change the grey-level order, and has no padding value. The headers of all images are read here (pixel data skipped) and summarised.

In [ ]:
import pydicom
from concurrent.futures import ThreadPoolExecutor
KEYS = ["PhotometricInterpretation", "BitsStored", "BitsAllocated", "PixelRepresentation", "SamplesPerPixel", "Rows", "Columns",
        "Modality", "RescaleSlope", "RescaleIntercept", "WindowCenter", "WindowWidth", "PixelPaddingValue", "TransferSyntax"]
def head(fp):
    ds = pydicom.dcmread(fp, stop_before_pixels=True)
    r = {k: (str(getattr(ds, k)) if hasattr(ds, k) else "absent") for k in KEYS if k != "TransferSyntax"}
    r["VOILUTSequence"] = "present" if "VOILUTSequence" in ds else "absent"
    r["ModalityLUTSequence"] = "present" if "ModalityLUTSequence" in ds else "absent"
    r["TransferSyntax"] = str(ds.file_meta.TransferSyntaxUID.name) if hasattr(ds, "file_meta") else "absent"
    return r
files = sorted(IMG_DIR.glob("*.dcm"))
t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    H = pd.DataFrame(list(ex.map(head, files)))
print(f"{len(H)} headers in {time.time() - t0:.0f} s")
rows = []
for k in H.columns:
    for v, n in H[k].value_counts().items():
        rows.append(dict(attribute=k, value=v, n=int(n)))
R14 = save_table(pd.DataFrame(rows), "R14_dicom_headers")
print(R14.to_string(index=False))

## 2 · Cross-patient reference by predicted class

In the main run every patient was paired with the previous patient of its processing shard (cyclic), so the partner is effectively random and may have a different predicted class. Each clean map was computed for the class predicted for that patient, so a Grad-CAM pair may explain two different classes. **R15** gives the mean cross-patient XFI for all pairs, for pairs with the same predicted class and for pairs with different predicted classes.

In [ ]:
rows = []
for M in MODELS:
    pred = main[M].drop_duplicates("Patient").set_index("Patient").pred_clean
    cp = base[M][base[M].Method.str.startswith("CrossPatient")].copy()
    cp["method"] = cp.Method.str.split("_").str[1]
    cp["same"] = cp.Patient.map(pred).values == cp.partner.map(pred).values
    for m in METHODS:
        q = cp[cp.method == m]
        for grp, s in [("all", q), ("same class", q[q.same]), ("different class", q[~q.same])]:
            e = ci(s.xfi.values)
            rows.append(dict(model=M, method=m, pairs=grp, n=len(s), mean_xfi=e[0], ci_low=e[1], ci_high=e[2]))
R15 = save_table(pd.DataFrame(rows), "R15_crosspatient_by_class")
print(R15.pivot_table(index=["model", "method"], columns="pairs", values="mean_xfi").round(3).to_string())

## 3 · Grad-CAM against Eigen-CAM without prediction flips

The paired comparison of each condition is repeated on the patients whose predicted class did not change under that condition (the flip does not depend on the explanation method, so the pairs stay complete). Under Gaussian noise L2 and L3 the classifiers assign almost every image to one class, so the remaining patients are those whose clean prediction already was that class; `share_kept` reports how many remain. Holm correction within each classifier, as in R2.

In [ ]:
def compare(a, b_, **kw):
    d = a - b_; mdn = ci(d, np.median); w = ST.wilcoxon_paired(a, b_)
    return dict(**kw, n=len(d), median_diff=mdn[0], median_ci_low=mdn[1], median_ci_high=mdn[2], p=w["p"], r=w["r"])
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    n_all = x.Patient.nunique(); sub = []
    for t in TYPES:
        for l in LEVELS:
            q = x[(x.Type == t) & (x.Level == l)]
            for subset, s in [("all", q), ("no flip", q[q.flip == 0])]:
                pv = s.pivot_table(index="Patient", columns="Method", values="xfi").dropna()
                if len(pv) < 10:
                    sub.append(dict(model=M, type=t, level=l, subset=subset, n=len(pv), share_kept=len(pv) / n_all)); continue
                sub.append(compare(pv.GradCAM.values, pv.EigenCAM.values, model=M, type=t, level=l, subset=subset, share_kept=len(pv) / n_all))
    sub = pd.DataFrame(sub)
    for subset in ["all", "no flip"]:
        k = (sub.subset == subset) & sub.p.notna()
        sub.loc[k, "p_holm"] = ST.holm(sub.loc[k, "p"].values)
    rows.append(sub)
R16 = save_table(pd.concat(rows, ignore_index=True), "R16_grad_vs_eigen_noflip")
R16["sign"] = np.where(R16.p_holm < 0.05, np.sign(R16.median_diff), 0)
chk = R16.pivot_table(index=["model", "type", "level"], columns="subset", values="sign")
print("sign of the significant median difference (+1 Grad more fragile, -1 Grad less fragile, 0 n.s.):")
print(chk.astype("Int64").to_string())
print("\nconditions where the conclusion changes:", int((chk["all"] != chk["no flip"]).sum()), "of", len(chk))

## 4 · Cluster-bootstrap CIs for the validity correlations

The Spearman correlations of XFI with the input SSIM, the input PSNR and $|\Delta\mathrm{IoU}|$ (positive patients) are computed over all patient-condition rows, as in R8. Because the nine rows of a patient are dependent, the CI resamples **patients** with replacement and keeps all their rows (cluster bootstrap).

In [ ]:
def cluster_spearman(df, a, b, B, seed):
    df = df[[ "Patient", a, b]].dropna()
    pats = df.Patient.unique(); pidx = {p: i for i, p in enumerate(pats)}
    g = df.Patient.map(pidx).values
    order = np.argsort(g, kind="stable"); g, xa, xb = g[order], df[a].values[order], df[b].values[order]
    start = np.searchsorted(g, np.arange(len(pats))); stop = np.searchsorted(g, np.arange(len(pats)), side="right")
    rng = np.random.default_rng(seed); est = spearmanr(xa, xb)[0]; bs = np.empty(B)
    for k in range(B):
        pick = rng.integers(0, len(pats), len(pats))
        ii = np.concatenate([np.arange(start[p], stop[p]) for p in pick])
        bs[k] = np.corrcoef(rankdata(xa[ii]), rankdata(xb[ii]))[0, 1]
    lo, hi = np.quantile(bs, [0.025, 0.975])
    return est, lo, hi
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)]
    for m in METHODS:
        s = x[x.Method == m]; sp = s[s.Label == 1]
        for name, df, col in [("input_ssim", s, "input_ssim"), ("input_psnr", s, "input_psnr"), ("abs_d_iou_pos", sp, "abs_d_iou")]:
            t0 = time.time(); e = cluster_spearman(df, "xfi", col, B_CORR, SEED)
            rows.append(dict(model=M, method=m, correlate=name, rho=e[0], ci_low=e[1], ci_high=e[2], n_rows=len(df), n_patients=df.Patient.nunique()))
            print(f"{M} {m:8s} {name:14s} rho {e[0]:+.3f} [{e[1]:+.3f}, {e[2]:+.3f}]  ({time.time() - t0:.0f} s)")
R17 = save_table(pd.DataFrame(rows), "R17_validity_ci")

## 5 · XFI by prediction outcome on the clean image

Reviewer 1 (comment c) asked for the fragility results to be stratified by whether the prediction is correct. The XFI never uses the label, but the outcome group can matter through the predicted class: it is the target of Grad-CAM, and it decides whether a perturbation that drives the classifier to one class causes a flip. Every patient is assigned once, from the clean prediction at the operating threshold: TP (label 1, predicted 1), FN (1, 0), TN (0, 0), FP (0, 1). The unit is the patient (mean XFI over the nine conditions). Per group: mean XFI per method with bootstrap CI, and the paired Grad-CAM minus Eigen-CAM median difference with the Wilcoxon test, Holm-corrected over the four groups of each classifier.

In [ ]:
GROUPS = ["TP", "FN", "TN", "FP"]
rows = []
for M in MODELS:
    x = main[M][main[M].Method.isin(METHODS)].copy()
    x["group"] = np.select([(x.Label == 1) & (x.pred_clean == 1), (x.Label == 1) & (x.pred_clean == 0),
                            (x.Label == 0) & (x.pred_clean == 0)], ["TP", "FN", "TN"], "FP")
    pv = x.pivot_table(index=["group", "Patient"], columns="Method", values="xfi", aggfunc="mean").dropna()
    sub = []
    for g in GROUPS:
        if g not in pv.index.get_level_values(0):
            continue
        q = pv.loc[g]; eg, ee = ci(q.GradCAM.values), ci(q.EigenCAM.values)
        r = compare(q.GradCAM.values, q.EigenCAM.values, model=M, group=g)
        sub.append(dict(r, grad_mean=eg[0], grad_ci_low=eg[1], grad_ci_high=eg[2], eigen_mean=ee[0], eigen_ci_low=ee[1], eigen_ci_high=ee[2]))
    sub = pd.DataFrame(sub); sub["p_holm"] = ST.holm(sub.p.values); rows.append(sub)
R18 = save_table(pd.concat(rows, ignore_index=True), "R18_xfi_by_outcome")
print(R18[["model", "group", "n", "grad_mean", "eigen_mean", "median_diff", "p_holm"]].round(4).to_string(index=False))

## Summary, artifacts and LOG entry

In [ ]:
from datetime import date
arts = pd.DataFrame([dict(path=str(Path(p).relative_to(ROOT)), bytes=Path(p).stat().st_size, sha256=sha256(p)) for p in PRODUCED])
arts.to_csv(OUT / "artifacts_phase6b.csv", index=False)
print(arts.to_string(index=False))
pi = R14[R14.attribute == "PhotometricInterpretation"]
cp = R15[R15.pairs != "all"].pivot_table(index=["model", "method"], columns="pairs", values="mean_xfi")
print(f"\n--- LOG entry ---\n## {date.today()} | Phase 6b sensitivity analyses (05_sensitivity), models {MODELS}")
print(f"- DICOM: {len(files)} headers; PhotometricInterpretation {dict(zip(pi.value, pi.n))}")
print(f"- Cross-patient XFI same vs different predicted class: {cp.round(3).to_dict('index')}")
print(f"- Grad vs Eigen without flips: conclusion changes in {int((chk['all'] != chk['no flip']).sum())} of {len(chk)} model-conditions")
print(f"- Validity correlations with patient-cluster bootstrap CIs (B={B_CORR}): R17")
if "R18" in globals():
    print("- XFI by clean outcome (patient level):", R18.set_index(["model", "group"])[["grad_mean", "eigen_mean"]].round(3).to_dict("index"))